#### RAG 심화
- 중복 문서 문제: 비슷한 내용의 청크가 여러 개 반환되어 컨텍스트 낭비 발생
- 검색: 시멘틱 유사도 만으로는 정확한 키워드 매칭이 어려움
- 구조적 질의 불가: "2024년 이후 계약 금액이 1억 이상인 제품 찾기" 같은 메타 필터 처리 불가
- 노이즈 청크: 관련성이 낮은 청크가 LLM에게 전달되면 환각 유발

임배딩 바꿔보고, 청크 크기도 바꿔보고

1. 
2. 유사도 검색
3. SelfQuery
4. BM25

In [1]:
%pip install lark
%pip install easyocr pymupdf
%pip install rank-bm25
%pip install cohere langchain-cohere sentence-transformers 
%pip install beautifulsoup4

  Using cached lark-1.3.1-py3-none-any.whl.metadata (1.8 kB)
Using cached lark-1.3.1-py3-none-any.whl (113 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached easyocr-1.7.2-py3-none-any.whl.metadata (10 kB)
  Using cached pymupdf-1.27.2.3-cp310-abi3-win_amd64.whl.metadata (24 kB)
  Using cached torch-2.12.0-cp312-cp312-win_amd64.whl.metadata (31 kB)
  Using cached torchvision-0.27.0-cp312-cp312-win_amd64.whl.metadata (5.5 kB)
  Using cached opencv_python_headless-4.13.0.92-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached scipy-1.17.1-cp312-cp312-win_amd64.whl.metadata (60 kB)
  Using cached numpy-2.4.6-cp312-cp312-win_amd64.whl.metadata (6.6 kB)
  Using cached pillow-12.2.0-cp312-cp312-win_amd64.whl.metadata (9.0 kB)
  Using cached scikit_image-0.26.0-cp312-cp312-win_amd64.whl.metadata (15 kB)
  Using cached python_bidi-0.6.10-cp312-cp312-win_amd64.whl.metadata (5.4 kB)
  Using cached pyyaml-6.0.3-cp312-cp312-win_amd64.whl.metadata (2.4 kB)
  Using cached shapely-2.1.2-cp312-cp312-win_amd64.whl.metadata (7.1 kB)
  Using cached pyclipper-1.4.0-cp312-cp312-win_amd64.whl.metadata (8.8 kB)
  Using cached ninja-1.13.0-py3-no


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached rank_bm25-0.2.2-py3-none-any.whl.metadata (3.2 kB)
Using cached rank_bm25-0.2.2-py3-none-any.whl (8.6 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached sentence_transformers-5.5.1-py3-none-any.whl.metadata (18 kB)
  Using cached fastavro-1.12.2-cp312-cp312-win_amd64.whl.metadata (6.0 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached pydantic_core-2.47.0-cp312-cp312-win_amd64.whl.metadata (6.6 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached tokenizers-0.23.1-cp310-abi3-win_amd64.whl.metadata (10 kB)
  Using cached types_requests-2.33.0.20260518-py3-none-any.whl.metadata (2.2 kB)
  Using cached cohere-5.21.1-py3-none-any.whl.metadata (3.6 kB)
  Using cached langchain_core-1.4.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached types_pyyaml-6.0.12.20260518-py3-none-any.whl.metadata (1.7 kB)
  Using cached transformers-5.9.0-py3-none-any.whl.metadata (33 kB)
  Using cached huggingface_hub-1.17.0-py3-none-any.whl.metadata (14 kB)
  Using cached scikit_learn-1.8.0-cp312-cp312-win_amd64.whl.met


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached beautifulsoup4-4.14.3-py3-none-any.whl.metadata (3.8 kB)
  Using cached soupsieve-2.8.4-py3-none-any.whl.metadata (4.6 kB)
Using cached beautifulsoup4-4.14.3-py3-none-any.whl (107 kB)
Using cached soupsieve-2.8.4-py3-none-any.whl (37 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
%pip install openai
%pip install langchain_openai
%pip install pypdf beautifulsoup4 youtube-transcript-api langchain-chroma faiss-cpu pdfplumber
%pip install langchain_text_splitters
%pip install chroma
%pip install langchain langchain-community langchain-ollama langchain-core python-dotenv
%pip install langchain-ibm ibm-watsonx-ai gradio
%pip install P

  Using cached openai-2.40.0-py3-none-any.whl.metadata (32 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached jiter-0.15.0-cp312-cp312-win_amd64.whl.metadata (5.3 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
Using cached openai-2.40.0-py3-none-any.whl (1.4 MB)
Using cached distro-1.9.0-py3-none-any.whl (20 kB)
Using cached jiter-0.15.0-cp312-cp312-win_amd64.whl (197 kB)
Using cached sniffio-1.3.1-py3-none-any.whl (10 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached langchain_openai-1.2.2-py3-none-any.whl.metadata (3.1 kB)
  Using cached tiktoken-0.13.0-cp312-cp312-win_amd64.whl.metadata (6.8 kB)
Using cached langchain_openai-1.2.2-py3-none-any.whl (99 kB)
Using cached tiktoken-0.13.0-cp312-cp312-win_amd64.whl (874 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached pypdf-6.12.2-py3-none-any.whl.metadata (7.2 kB)
  Using cached youtube_transcript_api-1.2.4-py3-none-any.whl.metadata (24 kB)
  Using cached langchain_chroma-1.1.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached faiss_cpu-1.14.2-cp312-cp312-win_amd64.whl.metadata (7.8 kB)
  Using cached pdfplumber-0.11.9-py3-none-any.whl.metadata (43 kB)
  Using cached defusedxml-0.7.1-py2.py3-none-any.whl.metadata (32 kB)
  Using cached chromadb-1.5.9-cp39-abi3-win_amd64.whl.metadata (5.1 kB)
  Using cached pdfminer_six-20251230-py3-none-any.whl.metadata (4.3 kB)
  Using cached pypdfium2-5.9.0-py3-none-win_amd64.whl.metadata (68 kB)
  Using cached cryptography-48.0.0-cp311-abi3-win_amd64.whl.metadata (4.3 kB)
  Using cached build-1.5.0-py3-none-any.whl.metadata (5.7 kB)
  Using cached pydantic_settings-2.14.1-py3-none-any.whl.metadata (3.4 kB)
  Using cached pybase64-1.4.3-cp312-cp312-win_amd64.whl.metadata (9.1 kB)
  Using cached uvicorn-0.48.0-py3-none-any.whl.metadata (6.7 kB)
  Us

...


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached langchain_ibm-1.0.11-py3-none-any.whl.metadata (6.3 kB)
  Using cached ibm_watsonx_ai-1.5.12-py3-none-any.whl.metadata (7.5 kB)
  Using cached gradio-6.15.2-py3-none-any.whl.metadata (17 kB)
  Using cached json_repair-0.59.10-py3-none-any.whl.metadata (19 kB)
  Using cached pandas-2.3.3-cp312-cp312-win_amd64.whl.metadata (19 kB)
  Using cached lomond-0.3.3-py2.py3-none-any.whl.metadata (4.1 kB)
  Using cached tabulate-0.10.0-py3-none-any.whl.metadata (40 kB)
  Using cached ibm_cos_sdk-2.14.3-py3-none-any.whl
  Using cached cachetools-7.1.4-py3-none-any.whl.metadata (5.5 kB)
  Using cached brotli-1.2.0-cp312-cp312-win_amd64.whl.metadata (6.3 kB)
  Using cached fastapi-0.136.3-py3-none-any.whl.metadata (27 kB)
  Using cached gradio_client-2.5.0-py3-none-any.whl.metadata (7.1 kB)
  Using cached groovy-0.1.2-py3-none-any.whl.metadata (6.1 kB)
  Using cached hf_gradio-0.4.1-py3-none-any.whl.metadata (428 bytes)
  Using cached pydub-0.25.1-py2.py3-none-any.whl.metadata (1.4 kB


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached p-1.5.0-py3-none-any.whl.metadata (9.5 kB)
  Using cached click-7.1.2-py2.py3-none-any.whl.metadata (2.9 kB)
  Using cached click_didyoumean-0.3.1-py3-none-any.whl.metadata (3.9 kB)
Using cached p-1.5.0-py3-none-any.whl (10 kB)
Using cached click-7.1.2-py2.py3-none-any.whl (82 kB)
Using cached click_didyoumean-0.3.1-py3-none-any.whl (3.6 kB)
  Attempting uninstall: click
    Found existing installation: click 8.4.1
    Uninstalling click-8.4.1:
      Successfully uninstalled click-8.4.1
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
huggingface-hub 1.17.0 requires click>=8.4.0, but you have click 7.1.2 which is incompatible.
typer 0.25.1 requires click>=8.2.1, but you have click 7.1.2 which is incompatible.

[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [41]:
from langchain_ollama import ChatOllama
from langchain_ibm import ChatWatsonx
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser, PydanticOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.chat_history import InMemoryChatMessageHistory, BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from pydantic import BaseModel, Field
from typing import Literal
from dotenv import load_dotenv
import os

from langchain_community.document_loaders import PyPDFLoader, CSVLoader, WebBaseLoader, DirectoryLoader
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_ibm import WatsonxEmbeddings
from langchain_chroma import Chroma
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI

from langchain_classic.chains.query_constructor.base import AttributeInfo
from langchain_classic.retrievers import EnsembleRetriever, ContextualCompressionRetriever,BM25Retriever
from langchain_classic.retrievers.self_query.chroma import ChromaTranslator
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_cohere import CohereRerank
from langchain_classic.retrievers.document_compressors import LLMChainExtractor, EmbeddingsFilter, DocumentCompressorPipeline



In [23]:
# .env 내용 가져오기
load_dotenv()

apiKey = os.getenv("WATSONX_API_KEY")
project_id = os.getenv("WATSONX_PROJECT_ID")
watsonx_ai_url = os.getenv("WATSONX_URL")
hf_token = os.environ["HF_TOKEN"]
COHERE_API_KEY = os.environ["COHERE_API_KEY"]

In [6]:
ollama_embedding = OllamaEmbeddings(model="nomic-embed-text-v2-moe")
watson_embedding = WatsonxEmbeddings(
    model_id="ibm/granite-embedding-278m-multilingual",
    url = f"{watsonx_ai_url}",
    api_key = f"{apiKey}",
    project_id=f"{project_id}"
)

# hugging_llm = ChatOpenAI(
#   base_url = "https://router.huggingface.co/v1",
#   api_key=hf_token,
#   model="Qwen/Qwen2.5-7B-Instruct:together",
#   temperature=0
# )


watson_llm = ChatWatsonx(
  model_id="ibm/granite-4-h-small",
  url=f"{watsonx_ai_url}",
  api_key = f"{apiKey}",
  project_id=f"{project_id}",
  max_tokens = 2000,
  params = {
    "temperature":0
  }
)
qwen_llm = ChatOllama(model="qwen3.5:4b", temperature=0)
exaone_llm = ChatOllama(model="exaone3.5:2.4b", temperature=0)


In [7]:
# pdf -> chunks 함수
def create_chunks_from_pdf(pdf_path, chunk_size=500, chunk_overlap=50):

  loader = PyPDFLoader(pdf_path)
  docs = loader.load()
  splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size,chunk_overlap=chunk_overlap)
  chunks = splitter.split_documents(docs)

  # 공백제거
  chunks = [chunk for chunk in chunks if chunk.page_content.strip()]
  return chunks

def create_vectorstore(chunks, embeddings, collection_name, persist_directory="./db/chroma_db"):
  return Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=persist_directory,
    collection_name=collection_name
  )

def create_retriever(vectorstore, search_type="similarity", k=3, fetch_k=20, lambda_mult=0.5):
  kwargs = {'k': k}

  if search_type=="mmr":
    kwargs['fetch_k'] = fetch_k
    kwargs['lambda_mult'] = lambda_mult

  return vectorstore.as_retriever(search_type=search_type, search_kwargs=kwargs)

def print_retrieved_docs(title, retriever, query):
  docs = retriever.invoke(query)

  print("\n"+"="*50)
  print(title)
  print("="*50)

  for i, doc in enumerate(docs):
    print(f"\n[chunk {i}]")
    print(doc.page_content)
    print(f"\nPage: {doc.metadata.get("page")}")

In [ ]:
### 1. 임베딩 모델, 청크 사이즈, 오버랩

In [12]:
chunks1 = create_chunks_from_pdf("./data/Summary of ChatGPTGPT-4 Research.pdf", 1000, chunk_overlap=100)
chunks2 = create_chunks_from_pdf("./data/Summary of ChatGPTGPT-4 Research.pdf", 100, chunk_overlap=10)

print(f"분할된 청크 수: {len(chunks1)}")
print(f"분할된 청크 수: {len(chunks2)}")

분할된 청크 수: 118
분할된 청크 수: 1216


In [13]:
vectorstore1 = create_vectorstore(chunks1, watson_embedding, collection_name="gpt_research_watson11")
vectorstore2 = create_vectorstore(chunks2, watson_embedding, collection_name="gpt_research_watson22")

watson1_retriever = create_retriever(vectorstore1)
watson2_retriever = create_retriever(vectorstore2)

query = 'where can i use ChatGPT?'

print_retrieved_docs("chunk=1000,overlap=100", watson1_retriever, query)
print_retrieved_docs("chunk=100,overlap=10", watson2_retriever, query)


chunk=1000,overlap=100

[chunk 0]
development.
2 Related work of ChatGPT
In this section, we review the latest research related to the application, ethics,
and evaluation of ChatGPT.
2.1 Application of ChatGPT
2.1.1 Question And Answering
In the education ﬁeld
ChatGPT is commonly used for question and answers testing in the edu-
cation sector. Users can use ChatGPT to learn, compare and verify answers
for diﬀerent academic subjects such as physics, mathematics, and chemistry,
4

Page: 3

[chunk 1]
of LLM for a variety of transportation tasks.
Nowadays, ChatGPT shows a wide range of applications in data visualiza-
tion, information extraction, data enhancement, quality assessment, and multi-
modal data processing.But there are also issues on how to further utilize hints
to eﬀectively interact with ChatGPT, lack of ability to process and analyze data
18

Page: 17

[chunk 2]
have been widely used for text classiﬁcation, recent advances in natural lan-
guage processing have led to the dev

In [8]:
vectorstore3 = create_vectorstore(chunks1, ollama_embedding, collection_name="gpt_research_watson33")
vectorstore4 = create_vectorstore(chunks1, watson_embedding, collection_name="gpt_research_watson44")

watson3_retriever = create_retriever(vectorstore3)
watson4_retriever = create_retriever(vectorstore4)

query = 'where can i use ChatGPT?'

print_retrieved_docs("ollama", watson3_retriever, query)
print_retrieved_docs("watson", watson4_retriever, query)

NameError: name 'chunks1' is not defined

vectorstore1 = create_vectorstore(chunks1, ollama_embedding, collection_name="gpt_research_watson3")
vectorstore2 = create_vectorstore(chunks2, watson_embedding, collection_name="gpt_research_watson4")

watson1_retriever = create_retriever(vectorstore1)
watson2_retriever = create_retriever(vectorstore2)

query = 'where can i use ChatGPT?'

print_retrieved_docs("chunk=1000,overlap=100", watson1_retriever, query)
print_retrieved_docs("chunk=1000,overlap=100", watson2_retriever, query)

### 2. MMR(Maximal Marginal Relevance) Retriever
- 관련성(Relevance)과 다양성(Diversity) 고려
- 법률 문서, 기술 매뉴얼 처럼 유사 내용이 반복되는 문서에 효과적
- 동작과정
  - 질문 -> 임배딩 -> 백터 스토어에서 질문과 유사한 상위 fetch_k(후보 문서) 추출
  - 후보문서에서 MMR 점수 계산 -> 가장 높은 문서 추출
  - 남은 후보에서 MMR 점수 계산 -> 높은 문서 추출
  - 추출한 높은 문서에서 최종 k 반환

In [11]:
chunks3 = create_chunks_from_pdf("./data/2026 상 삼성전자 DX부문 직무기술서.pdf")
vectorstore1 = create_vectorstore(chunks3, watson_embedding, collection_name="samsung_watson1",persist_directory="./db/watson_chroma2")

mmr_retriever = create_retriever(vectorstore1,search_type="mmr", k=5)
similarity_retriever = create_retriever(vectorstore1, search_type="similarity")

query = '직무 분석'
print_retrieved_docs("MMR", mmr_retriever, query)
print_retrieved_docs("Similarity", similarity_retriever, query)


MMR

[chunk 0]
•
•
•
•
•
•
•

Page: 19

[chunk 1]
•
•
•
•
•
•
•
•
•

Page: 22

[chunk 2]
해외영업
고객과 시장
 제품에 대한 이해를 바탕으로 시장 수요와 경쟁환경을 분석하여 국가
 거래선별 목표 설정
영업전략 수립
 신규 제품
 영업 채널을 발굴하고 판매전략 수립 및 실행을 통해 매출 극대화에
기여합니다

Page: 18

[chunk 3]
S/W개발
소프트웨어 기술에 대한 전문적인 지식을 기반으로 창의적이고 분석적인 사고를 통해 신기술을
선도하고 당사 제품에 반영함으로써 제품 및 솔루션의 혁신적인 가치를 창출합니다

Page: 5

[chunk 4]
•
 ∙
•
•
•
•
•
•
•
•
•
∙

Page: 29

Similarity

[chunk 0]
•
•
•
•
•
•
•

Page: 19

[chunk 1]
•
•
•
•
•
•
•
•
•

Page: 22

[chunk 2]
•
•
•
•
•
•
•
•
•

Page: 27


### 3. SelfQuery Retriever
- 자연어 질문을 분석하여 시멘틱 검색쿼리와 메타데이터필터를 LLM이 자동으로 생성하게 하는 고급 Retriever
- 질문: 2023년 이후 계약금이 1억 이상인 계약 찾아줘 -> LLM
  - 시멘틱 검색쿼리: 계약
  - filter: {year >= 2023, 계약금액 >= 100000}
- 메타데이터 작업이 필요

In [28]:
!pip install lark


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
docs = [
    Document(
        page_content="삼성전자 제품 마케팅 직무입니다.",
        metadata={
            "year":2025,
            "department":"marketing"
        }
    ),
    Document(
        page_content="AI 연구 개발 직무입니다.",
        metadata={
            "year":2024,
            "department":"ai"
        }
    ),
    Document(
        page_content="백엔드 개발 직무입니다.",
        metadata={
            "year":2025,
            "department":"developer"
        }
    ),
]


metadata_field_info = [
    AttributeInfo(name="year", description="문서 작성 연도", type="integer"),
    AttributeInfo(name="department", description="직무 부서", type="string"),
]

document_content_description = "회사 내부 문서 및 직무 자료"


In [35]:
try:
    import lark
    print("✅ lark 패키지가 정상적으로 설치되었고, 현재 환경에서 인식 가능합니다!")
    print(f"설치된 버전: {lark.__version__}")
except ImportError:
    print("❌ 에러: 여전히 lark 패키지를 불러올 수 없습니다. 환경이 꼬여있습니다.")

✅ lark 패키지가 정상적으로 설치되었고, 현재 환경에서 인식 가능합니다!
설치된 버전: 1.3.1


In [10]:


vectorestore1 = create_vectorstore(docs,watson_embedding,collection_name="selfquery",persist_directory="./db/watson_chroma")

self_query_retriever = SelfQueryRetriever.from_llm(
    llm=watson_llm,
    vectorstore=vectorestore1,
    document_contents=document_content_description,
    metadata_field_info=metadata_field_info,
    verborse=True,
    enable_limit=True,
    structured_query_translator=ChromaTranslator()
)



self_query_retriever.invoke("2024년 이후 ai 부서 직무를 찾아줘")


[Document(id='c5c3fe43-3529-44b7-960d-4ff1e60e49e3', metadata={'year': 2024, 'department': 'ai'}, page_content='AI 연구 개발 직무입니다.'),
 Document(id='8a2e8e05-c6d5-4b81-8c37-02c37338b6a7', metadata={'year': 2024, 'department': 'ai'}, page_content='AI 연구 개발 직무입니다.'),
 Document(id='69cd1794-2b2b-431f-a695-f462dc2cfc4a', metadata={'year': 2024, 'department': 'ai'}, page_content='AI 연구 개발 직무입니다.'),
 Document(id='39a4c24d-12dc-4403-a718-6504e3d22495', metadata={'department': 'ai', 'year': 2024}, page_content='AI 연구 개발 직무입니다.')]

In [23]:
# splitter, chunk 끝난 상태
# create_chunks_from_pdf
docs = [
    Document(
        page_content="수분 가득한 히알루론산 세럼으로 피부 속 깊은 곳까지 수분을 공급합니다.",
        metadata={
            "year":2024,
            "category":"스킨케어",
            "user_rating":4
        }
    ),
     Document(
        page_content="24시간 지속되는 매트한 피니시의 파운데이션, 모공을 커버하고 자연스러운 피부 표현 가능",
        metadata={
            "year":2023,
            "category":"메이크업",
            "user_rating":3
        }
    ),
     Document(
        page_content="식물성 성분으로 만든 저자극 클렌징 오일, 메이크업 노폐물을 부드럽게 제거합니다.",
        metadata={
            "year":2023,
            "category":"클렌징",
            "user_rating":5
        }
    ),
     Document(
        page_content="비타민C함유 브라이트닝 크림, 칙치한 피부톤을 환하게 밝혀줍니다",
        metadata={
            "year":2023,
            "category":"스킨케어",
            "user_rating":2
        }
    ),
     Document(
        page_content="롱래스팅 립스틱, 선명한 발색과 촉촉한 사용감으로 하루종일 편안하게 사용 가능합니다.",
        metadata={
            "year":2024,
            "category":"메이크업",
            "user_rating":4
        }
    ),
     Document(
        page_content="자외선 차단 기능이 있는 톤업 선크림, spf50+/pa+++ 높은 자외선 차단 지수로 피부를 보호합니다.",
        metadata={
            "year":2025,
            "category":"썬케어",
            "user_rating":5
        }
    )
]

# 메타데이터 필드 정보 생성
metadata_field_info = [
    AttributeInfo(
        name="year", 
        description="화장품 출시 연도", 
        type="integer"
    ),
     AttributeInfo(
        name="category", 
        description="화장품 카테고리 ['스킨케어', '메이크업', '클렌징', '선케어']", 
        type="string"
    ),
     AttributeInfo(
        name="user_rating", 
        description="화장품 평점", 
        type="integer"
    )
]

document_content_description = "화장품 제품 정보"

vectorstore = create_vectorstore(docs, watson_embedding, collection_name="selfquery2",persist_directory="./db/watson_chroma")
self_query_retriever = SelfQueryRetriever.from_llm(
  llm=watson_llm,
  vectorstore=vectorstore,
  document_contents=document_content_description,
  metadata_field_info=metadata_field_info,
  verbose=True,
  enable_lint=True,
  structured_query_translator=ChromaTranslator()
)

ImportError: Cannot import lark, please install it with 'pip install lark'.

In [ ]:
self_query_retriever.invoke("2024년 이후로 평점이 4 이상인 제품을 추천해줘")

In [ ]:
self_query_retriever.invoke("카테고리가 썬케어인 상품 푸펀해줘")

In [ ]:
# 이미지라서 못 읽고 불릿만 가져옴
pdf_path="./data/2026 상 삼성전자 DX부문 직무기술서.pdf"
chunks = create_chunks_from_pdf(pdf_path)
for i in range(2):
    print("="*50)
    print(chunks[i].page_content[:500])

회로개발
회로 기술을 기반으로 삼성전자 제품 및 솔루션의 혁신적인 가치를 창출합니다
•
•
•
•
•
•
•
•
•
•
•
•
•
•


In [ ]:
!pip install easyocr pymupdf
# easyocr 이미지 -> 텍스트
# pymupdf 

  Using cached torch-2.12.0-cp312-cp312-win_amd64.whl.metadata (31 kB)
  Using cached setuptools-81.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/2.9 MB ? eta -:--:--
   ---------------------------------------- 2.9/2.9 MB 55.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/19.2 MB ? eta -:--:--
   ------------- -------------------------- 6.3/19.2 MB 29.6 MB/s eta 0:00:01
   ------------------------- -------------- 12.3/19.2 MB 28.6 MB/s eta 0:00:01
   ---------------------------------------- 19.2/19.2 MB 31.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 4.0/4.0 MB 47.6 MB/s eta 0:00:00
Using cached torch-2.12.0-cp312-cp312-win_amd64.whl (123.0 MB)
   -------


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [36]:
# PDF를 이미지로 변환
import fitz
pdf = fitz.open(pdf_path)
for page_num in range(len(pdf)):
    page = pdf[page_num]
    pix = page.get_pixmap(dpi=300)
    pix.save(f"page_{page_num}.png")

In [20]:
# 이미지에서 텍스트를 추축
import easyocr

reader = easyocr.Reader(['ko','en'])
result = reader.readtext("page_19.png",detail=0,paragraph=True)
print("\n".join(result))

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.
c:\source\ollama\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


DA사업부(수원 근무) 해외영업 B2C 영업
포지션 소개 Job Overview
글로벌 시장과 소비자에 대한 이해틀 바탕으로 생활가전 제품의 판매 전락올 수립하고 매출 확대와 브랜드 리더십올 제고합니다:
수행업무 Job Details
담당 지역의 시장과 고객 특성, 판매 데이터들 분석하여 매출과 손의 극대화틀 위한 제품 가격유통 마켓팅올 아우르는 장단기 판매 전락올 수립합니다. 수립든 판매 전락올 기반으로 담당 법인과 현업하여 적기 공급올 위한 오퍼레이선올 지원하고 현지 법인의 실행 계획과 진척올 관리 및 개선합니다.
자격요건 Requirements
영어로 해외 자료 조사 및 커류니키이선이 가능하신 분 Global 이문화에 대한 이해도가 높으신 분 팀위크와 협업 능력올 보유하신 분
우대사항 Preferences
직무와 연관된 대내외 활동 경험올 보유하신 분 제2외국어 회화 역량울 보유하신 분
커리어 비전 Career Vision
담당 지역의 언어와 문화 습득올 포함한 글로벌 역량울 강화할 수 있으려 고객시장 유통 특성 등 전문 지식 습득과 법인 관리 및 개선올 통한 영업 실무 경험올 쌍울 수 있습니다. 삼성전자 해외영업 주재원으로서 현지 법인에서 근무하여, 매출 확대와 브랜드 리더십올 공고히 하는 현장 경험올 익히려 글로벌 영업 전문가로 성장할 수 있습니다:
'글로벌올 무대로 DA의 새로운 가능성울 열어갈 당신에게"
시장과 소비자에 대한 이해틀 바탕으로
생활가전 브랜드틀 최고의 자리로 이끌어 칼 인재틀 모십니다.


In [21]:
import json
pages = []

for page_num in range(31):
    image_path = f"page_{page_num}.png"
    result = reader.readtext(image_path,detail=0,paragraph=True)
    text = "\n".join(result)
    pages.append({"page": page_num+1, "text": text})

with open("./data/samsung_dx_ocr.json", "w", encoding="utf-8") as f:
    json.dump(pages, f, ensure_ascii=False, indent=2)
    



c:\source\ollama\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\source\ollama\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\source\ollama\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\source\ollama\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\source\ollama\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' a

### 4. BM25
- 유사도 검색은 의미적 유사성 위주
- 정확한 제품명, 코드, 버전, 고유명사 등의 키워드 매칭에는 약함
- 유사도 검색의 단점 보안 (semantic + sparse)
- 예 
    - 환불 가능한 기간이 어떻게 되나요? => 유사도 검색이 유리
    - ERR_CONNECTION 오류 => 키워드 검색

In [30]:
%pip install rank-bm25

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [31]:

import json

with open("./data/samsung_dx_ocr.json", "r", encoding="utf-8") as f:
    pages = json.load(f)

# load
docs = []
for page in pages:
    docs.append(
        Document(
            page_content=page['text'],
            metadata={
                "page": page['page'],
                "source": "samsung_dx_ocr"
            }

        )
    )

In [33]:
len(docs)

# chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(docs)

vectorstore = create_vectorstore(chunks=chunks, embeddings=watson_embedding, persist_directory="./db/chroma_db", collection_name="samsung_dx")

# similarity - 단순 검색 
vectorstore.similarity_search("",k=3)

[Document(id='67bc019e-b26f-41be-8c57-ecf1784ef334', metadata={'page': 18, 'source': 'samsung_dx_ocr'}, page_content='자격요건 Requirements\n해외 법인 고객과의 원활한 커류니키이선 및 프레전테이선이 가능한 외국어 역량울 보유하신 분 다양한 채널 및 데이터틀 기반으로 시장 상황과 트랜드틀 분석할 수 짓는 역량울 보유하신 분 소비자 라이프 스타일, 제품 구매 양식 등 소비자에 대한 이해도틀 보유하신 분 최신 Tech 동향 및 Gen-Z틀 포함한 새로운 문화 Trendol 대한 이해도가 높으신 분\n우대사항 Preferences\n마켓팅/커유니키이선 관련 전공올 하신 분 또는 이에 상응하는 전공지식올 보유하신 분 Smartphone Tablet Wearable 기술에 대한 관심 및 이해도가 높으신 분\n커리어 비전 Career Vision'),
 Document(id='79e671f6-50ab-4919-9db9-c5f14db0e146', metadata={'page': 18, 'source': 'samsung_dx_ocr'}, page_content='자격요건 Requirements\n해외 법인 고객과의 원활한 커류니키이선 및 프레전테이선이 가능한 외국어 역량울 보유하신 분 다양한 채널 및 데이터틀 기반으로 시장 상황과 트랜드틀 분석할 수 짓는 역량울 보유하신 분 소비자 라이프 스타일, 제품 구매 양식 등 소비자에 대한 이해도틀 보유하신 분 최신 Tech 동향 및 Gen-Z틀 포함한 새로운 문화 Trendol 대한 이해도가 높으신 분\n우대사항 Preferences\n마켓팅/커유니키이선 관련 전공올 하신 분 또는 이에 상응하는 전공지식올 보유하신 분 Smartphone Tablet Wearable 기술에 대한 관심 및 이해도가 높으신 분\n커리어 비전 Career Vision'),
 Document(id='e6c46e61-13a2-434a-86fb-622ea2fa

In [34]:
# similarity - 단순 검색 
# vectorstore.similarity_search("",k=3)

# 유사도 검색(의미)
# as_retriever:  확장하고 싶을 때 더 편하고 / LangChang에서 제공
dens_retriever = create_retriever(vectorstore)

# 키워드 검색(법령, 제품명)
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 4

# 앙상블 리트리버 - 유사도 + 키워드
# weight: 검색 전략 0.4:0.6 dense:bm25
hybrid_retriever = EnsembleRetriever(retrievers=[dens_retriever,bm25_retriever], weight=[0.5,0.5])

query = "시스템 소프트웨어 자격요건에 운영체제 개념도 포함되어 있어?"
# search_docs = hybrid_retriever.invoke(query)
print_retrieved_docs("hybrid", hybrid_retriever, query)


hybrid

[chunk 0]
자격요건 Requirements
컴퓨터 전기 전자, 기계 로봇공학 등 관련 전공올 하신 분 운영체제 기본 개념에 대한 이해도틀 보유하신 분 요구사항울 분석하여 소프트웨어들 구조적으로 설계 및 구현하는 역량울 보유하신 분 다양한 분야의 엔지니어와 적극적으로 소통하여 협업할 수 짓는 역량울 보유하신 분
우대사항 Preferences
CIC++ 기반 시스템 프로그래망 역량울 보유하신 분 Linux 기반 임베디드 시스템 개발 경험올 보유하신 분(Kernel, Device Driver, BSP 등) CAN EtherCAT, UDP 등 하드웨어 통신 프로토록 활용 경험 보유하신 분 ROS2 기반 로보텍스 프로적트 수행 경험올 보유하신 분 Git 기반 협업 및 CICD 환경에서의 개발 경험올 보유하신 분
커리어 비전 Career Vision

Page: 7

[chunk 1]
미래로봇추진단(서울 근무) S/W개발 시스템 소프트웨어
포지션 소개 Job Overview
휴머노이드 로봇의 실시간 제어 시스템 및 소프트웨어 플렉품올 개발하는 직무입니다: 운영체제 환경 구성, 디바이스 드라이버, 실시간 제어 프레임위크 등 로봇 동작의 핵심 기반이 되는 소프트웨어름 설계 및 개발하다, 하드웨어 설계 조직 및 Al 연구 조직과 긴밀히 현업합니다:
수행업무 Job Details
휴머노이드 로봇의 실시간 제어 프레임위크 설계하고 개발합니다: 모터; 센서 등 로봇 하드웨어 제어름 위한 디바이스 드라이버 및 하드웨어 추상화 계층올 개발합니다. 로봇 제어용 운영체제(Linux 기반 실시간 OS 등) 환경올 구성하고 시스템 성능올 최적화합니다: 유관 부서와 협업하여 보행 조작 등 Al 기능과 제어 시스템 간 연동 미들웨어름 개발합니다:
자격요건 Requirements

Page: 7

[chunk 2]
커리어 비전 Career Vision
휴머노이드 로봇의 초기 개발 단계부터 참여하여 핵심 개발자로 성장할 수 있습니다: 실시간 제어; 시스템 아키택처 , 로봇 미

### ReRanking & Contextual Compression
- 질문 + 결과 -> 다시 한 번 리트리버 -> 다시 순위 
- 유사도 검색(Bi-Encoder): 쿼리(질문)와 문서를 각각 벡터로 변환한 후 계산
- Cross-Encoding: 쿼리 + 문서 쌍으로 벡터로 변환
    - 정확도가 매우 높음 / 매우 느리다

In [ ]:
%pip install cohere langchain-cohere sentence-transformers 


In [40]:
# 기본검색(유사도 검색)
from langchain_cohere import CohereRerank

base_retriever = create_retriever(vectorstore, k=20)
query = "원격근무 정책은 어떻게 되나요?"
docs = base_retriever.invoke(query)

# 20개 후보군을 대상으로 reranking
# ContextualCompressionRetriever 질문에 관련한 한, 두문장만 압축시켜 보내는 방법
reranker = CohereRerank(model="rerank-v4.0-pro", top_n=5)
compression_retriever = ContextualCompressionRetriever(base_compressor=reranker, base_retriever=base_retriever)

# 최종
# docs = compression_retriever.invoke(query)
print_retrieved_docs("Rerank", compression_retriever, query)



Rerank

[chunk 0]
Ver. 2.89
Ver. 2.89

Page: 41

[chunk 1]
Ver. 2.89
Ver. 2.89

Page: 0

[chunk 2]
· 오만찬비
· 셔틀버스 운영비
회의규모 100명 이상
(해외참가자 50명 이상)
국내회의 도외참가자 1,000명 이상
 - 기업회의 및 인센티브 투어 개최지원
구  분 유형 분류 체류기간 지원내용
국제회의 기업회의 개최 목적 단체
*일정표상 4시간 이상 회의 포함 제주 2박
이상
· 행사장 임대료
· 기념품 구입비
· 오만찬비
· 차량 임차비
· 환영(환대)행사 진행비
국내회의 인센티브(포상) 목적 단체
*기업체에서 경비 부담
4.기타지원: 제주 유니크베뉴 및 MICE 연계상품 활용 인센티브, 답사지원 등
5.문의사항: 제주관광공사 마이스뷰로팀 문의
MICE관광
MICE행사개최의최적지제주
NoVisa.NoTax의대한민국가장편리한개최지
대한민국유일24개국제외한모든국가무비자입국,30일 체류가능
최적의MICE인프라
· 제주전역세계적수준의서비스를제공하는특급호텔6,000여실및 다양한 등급의71,000
여실객실보유

Page: 36

[chunk 3]
· 유네스코인류무형문화유산‘제주해녀문화’등재(‘16.11.30)
국제안전도시
· 세계보건기구(WHO)국제안전도시3회연속지정(2017년,2012년,2007년)
· UN안전담당관의UN안전지침에의거한점검후 “국제회의최적지”격찬
70JEJU   ISL AND   POCKET   BOOK —71
Information

Page: 36

[chunk 4]
1. 유치지원: 국제회의를 제주로 유치하는 단계에서 지원
기준 지원내용
참가자 100명이상, 해외참가자
3개국 50명 이상, 2일 이상
· 전차대회(유관대회) 홍보부스 제작 및 운영비
·  유치제안서 및 온오프라인 홍보물(PT, 인쇄물, 영상, 
기념품 등)  
·  해외공식연회(KoreanNight·Lunch)
·  회의 유치 직접적인 활동 참가자 항공 (2인 이내)
2. 홍보지원: 참가자 증대를 위한 

### LLMChainExtractor
- 질문+문서를 같이 LLM에게 보내서 질문과 관련된 내용만 추출
- 현재 chunks 안에 있는 내용을 줄여내는 것

In [51]:
# 문서로드 / 청크 추출
chunks = create_chunks_from_pdf("./data/제주관광가이드.pdf")
# indexing - vectorstore 저장
vectorstore = create_vectorstore(chunks, watson_embedding, collection_name="jeju_guid2")
# 질의
query = "생활 속 제주어에서 엄불랑하다는 무슨 뜻이야?"
base_retriever = create_retriever(vectorstore=vectorstore,k=20)
docs= base_retriever.invoke(query)
for doc in docs:
    print(doc.page_content[:500])
    print("-"*10)

생활 속 제주어
제주는 타 지역보다 한국어의 고형(古形)을 많이 유지하고 있는 동시에 
제주도만의 고유한 어휘나 문법적 특성을 가지고 있다.
다른 지역 사람이 못 알아듣는 제주어
제주어 뜻풀이
솔쩨기 살짝
안네다 드리다
베지근허다 입안에 기름기가 감돌아 맛이 있다.
엄불랑허다 어마어마하다
코시롱허다 고소하다
산도록허다 시원하다  예) 물이 산도록헌 게 좋다.
두령청이 우두망찰
무사 왜
영, 경, 정 이렇게, 그렇게, 저렇게
게메 글쎄
인사말
제주어 뜻풀이
펜안허우꽈? 편안(안녕)하십니까?
제주도 오난 어떵허우꽈? 제주도에 오니 어떠십니까?
차말로 좋수다. 참말로 좋습니다.
공기도 마고, 산이영 바다잉여 마딱 좋은게마씀 공기도 맑고, 산이랑 바다랑 모두 좋네요.
서울 갈 때랑 하영 다앙 갑서. 서울 갈 때는 많이 담아서 가십시오.
게메양. 경 헤시민 얼마나 좋코마씀? 글쎄요. 그렇게 했으면 얼마나 좋겠습니까?
식당에서
제주어 뜻풀이
무신걸 먹으코? 무엇을 먹을까?
----------
생활 속 제주어
제주는 타 지역보다 한국어의 고형(古形)을 많이 유지하고 있는 동시에 
제주도만의 고유한 어휘나 문법적 특성을 가지고 있다.
다른 지역 사람이 못 알아듣는 제주어
제주어 뜻풀이
솔쩨기 살짝
안네다 드리다
베지근허다 입안에 기름기가 감돌아 맛이 있다.
엄불랑허다 어마어마하다
코시롱허다 고소하다
산도록허다 시원하다  예) 물이 산도록헌 게 좋다.
두령청이 우두망찰
무사 왜
영, 경, 정 이렇게, 그렇게, 저렇게
게메 글쎄
인사말
제주어 뜻풀이
펜안허우꽈? 편안(안녕)하십니까?
제주도 오난 어떵허우꽈? 제주도에 오니 어떠십니까?
차말로 좋수다. 참말로 좋습니다.
공기도 마고, 산이영 바다잉여 마딱 좋은게마씀 공기도 맑고, 산이랑 바다랑 모두 좋네요.
서울 갈 때랑 하영 다앙 갑서. 서울 갈 때는 많이 담아서 가십시오.
게메양. 경 헤시민 얼마나 좋코마씀? 글쎄요. 그렇게 했으면 얼마나 좋겠습니까?
식당에서
제주어 뜻풀이
무신걸 먹으코? 무엇을 먹을까?
----------


In [39]:
# 답변만 축약

# 문서의 양 감소
extractor = LLMChainExtractor.from_llm(watson_llm)
compression_retriever = ContextualCompressionRetriever(base_compressor=extractor, base_retriever=base_retriever)
docs = compression_retriever.invoke(query)
for doc in docs:
    print(doc.page_content[:500])
    print("-"*10)

엄불랑허다 어마어마하다
----------


### Embedding Filter
- 임계값을 기준으로 미달한 문서 제외
- 

In [52]:
embedding_filter = EmbeddingsFilter(embeddings=watson_embedding, similarity_threshold=0.8)
pipeline = DocumentCompressorPipeline(transformers=[embedding_filter, extractor])
compression_retriever = ContextualCompressionRetriever(base_compressor=pipeline, base_retriever=base_retriever)
docs = compression_retriever.invoke(query)
print("-----")
for doc in docs:
    print(doc.page_content[:500])
    print("="*10)

-----
엄불랑허다 어마어마하다
엄불랑허다 어마어마하다


In [53]:
filter_docs = embedding_filter.compress_documents(docs, query)
len(filter_docs)

0

#### RAG 성능 처리
1. 문서 전처리
- chunk 최적화, metadata 추가

2. Retrieval(검색) 개선
- MMR, BM25, Hybrid, SelfQuery

3. Retrieval 후 처리
- Rerank, EmbeddingFilter, LLMChainExtractor, ContextualCompressionRetriever

In [ ]:
# 1. 한양대대학원캠퍼스가이드.pdf 로드 후 chunk_size=500, overlap=50 split 한 후 청크 사이즈 확인
# 2. 청크 내용 확인

# pdf -> 이미지, 이미지 -> 텍스트 인식
# 벡터스토어에 저장 collection_name="hanyang_campus"

# 1
pdf_path = "./data/한양대 대학원 캠퍼스 가이드.pdf"
image_prefix="page_hy_"
chunks = create_chunks_from_pdf(pdf_path=pdf_path, chunk_size=500, chunk_overlap=50)
print(len(chunks))

# PDF를 이미지로 변환
pdf = fitz.open(pdf_path)
for page_num in range(len(pdf)):
    page = pdf[page_num]
    pix = page.get_pixmap(dpi=300)
    pix.save(f"page_hy_{page_num}.png") 



In [ ]:
# 이미지에서 텍스트를 추출
reader = easyocr.Reader(['ko','en'])
pages = []

for page_num in range(len(pdf)):
    image_path = f"page_hy_{page_num}.png"
    result = reader.readtext(image_path,detail=0,paragraph=True)
    text = "\n".join(result)
    pages.append({"page": page_num+1, "text": text})

with open(f"./data/page_hy_pdf.json", "w", encoding="utf-8") as f:
    json.dump(pages, f, ensure_ascii=False, indent=2)
    


Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.
c:\source\ollama\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\source\ollama\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\source\ollama\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\source\ollama\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\

In [ ]:

with open("./data/samsung_dx_ocr.json", "r", encoding="utf-8") as f:
    pages = json.load(f)

# load
docs = []
for page in pages:
    docs.append(
        Document(
            page_content=page['text'],
            metadata={
                "page": page['page'],
                "source": "samsung_dx_ocr"
            }

        )
    )


splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(docs)

vectorstore = create_vectorstore(chunks=chunks, embeddings=watson_embedding, persist_directory="./db/chroma_db", collection_name="hy_pdf")



In [ ]:
# 검색
hy_vectorstore = Chroma(persist_directory="./db/chroma_db", embedding_function=watson_embedding, collection_name="hy_campus")